In [ ]:
# Sentiment + Trust — Full Training Notebook
# Only run these installs if you need them (comment out on machines that already have packages).

!pip install transformers==4.29.0 datasets sentence-transformers scikit-learn pandas numpy -q
!pip install kaggle -q


In [ ]:
import os
import shutil
import zipfile
import glob
import pandas as pd
import numpy as np
import re
import torch
from datasets import Dataset
from transformers import (DistilBertTokenizer, DistilBertForSequenceClassification,
                          Trainer, TrainingArguments)
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pickle
from IPython.display import display


In [ ]:
def load_amazon_reviews(path='Reviews.csv'):
    cols = ["Id","ProductId","UserId","ProfileName",
            "HelpfulnessNumerator","HelpfulnessDenominator",
            "Score","Time","Summary","Text"]
    df = pd.read_csv(path, names=cols, header=0, encoding='utf-8-sig')
    # remove non-ascii text
    df = df[df['Text'].apply(lambda x: all(ord(c) < 256 for c in str(x)))]
    df = df[df.Score != 3]             # drop neutral 3-star
    df['label'] = (df.Score > 3).astype(int)
    return df

# If you have Reviews.csv in the working directory:
if os.path.exists('Reviews.csv'):
    df = load_amazon_reviews('Reviews.csv')
    print("Loaded Reviews.csv from local folder")
else:
    print("Reviews.csv not found. Please download it manually or use Kaggle API as described in README.")
    df = pd.DataFrame(columns=["Id","ProductId","UserId","ProfileName",
                               "HelpfulnessNumerator","HelpfulnessDenominator",
                               "Score","Time","Summary","Text","label"])


In [ ]:
display(df.head())
print(f"Dataset shape: {df.shape}")


In [ ]:
def train_bert_sentiment(data, num_epochs=3, max_len=128, batch_size=16):
    print("Training BERT sentiment model...")
    tok = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
    ds  = Dataset.from_pandas(data[['Text','label']].reset_index(drop=True))

    def tokenize_function(examples):
        return tok(examples['Text'], padding='max_length', truncation=True, max_length=max_len)

    ds_tok = ds.map(tokenize_function, batched=True)
    split = ds_tok.train_test_split(test_size=0.2, seed=42)

    model = DistilBertForSequenceClassification.from_pretrained(
                'distilbert-base-uncased', num_labels=2)

    args = TrainingArguments(
        output_dir='./results',
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        report_to='none'
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=split['train'],
        eval_dataset=split['test']
    )

    trainer.train()
    return model, tok, trainer


In [ ]:
# WARNING: Full training can be slow. Use .head(n) for quick runs.
if len(df) > 0:
    sample_for_training = df.head(5000)  # change or remove .head() to use more
    sentiment_model, sentiment_tok, trainer = train_bert_sentiment(sample_for_training)
    eval_results = trainer.evaluate()
    print("Evaluation Results:", eval_results)
else:
    print("No data to train on. Add Reviews.csv in the working directory.")


In [ ]:
class UserTrustScorePredictor:
    def __init__(self):
        self.model = RandomForestClassifier(n_estimators=100, random_state=42)
        self.scaler = StandardScaler()
        self.feature_names = []

    def extract_user_features(self, user_reviews_df):
        features = {}
        features['total_reviews'] = len(user_reviews_df)
        ratings = user_reviews_df['Score'].values
        features['rating_variance'] = np.var(ratings)
        features['extreme_rating_pct'] = np.mean((ratings == 1) | (ratings == 5)) * 100
        features['avg_rating'] = np.mean(ratings)
        features['rating_consistency'] = 1 - (np.std(ratings) / 5.0)
        review_lengths = user_reviews_df['Text'].str.len()
        features['avg_review_length'] = review_lengths.mean()
        features['review_length_variance'] = review_lengths.var()
        all_text = ' '.join(user_reviews_df['Text'].astype(str))
        words = re.findall(r'\b\w+\b', all_text.lower())
        features['vocabulary_diversity'] = len(set(words)) / max(len(words), 1)
        if 'HelpfulnessNumerator' in user_reviews_df.columns:
            helpful_votes = user_reviews_df['HelpfulnessNumerator'].sum()
            total_votes = user_reviews_df['HelpfulnessDenominator'].sum()
            features['helpfulness_ratio'] = helpful_votes / max(total_votes, 1)
            features['avg_helpful_votes'] = helpful_votes / max(features['total_reviews'], 1)
        else:
            features['helpfulness_ratio'] = 0
            features['avg_helpful_votes'] = 0
        first_person_pronouns = ['i', 'me', 'my', 'myself', 'we', 'us', 'our']
        total_pronouns = 0
        total_words = 0
        for text in user_reviews_df['Text']:
            if pd.isna(text):
                continue
            words = re.findall(r'\b\w+\b', str(text).lower())
            total_words += len(words)
            total_pronouns += sum(1 for word in words if word in first_person_pronouns)
        features['first_person_usage'] = total_pronouns / max(total_words, 1) * 100
        return features

    def create_trust_labels(self, user_features_df, threshold_percentile=70):
        trust_score = (
            user_features_df['helpfulness_ratio'] * 0.4 +
            user_features_df['rating_consistency'] * 0.3 +
            user_features_df['vocabulary_diversity'] * 0.2 +
            (user_features_df['first_person_usage'] / 100) * 0.1
        )
        threshold = np.percentile(trust_score, threshold_percentile)
        return (trust_score >= threshold).astype(int)

    def train_trust_model(self, reviews_df):
        user_features_list, user_ids = [], []
        for user_id, user_reviews in reviews_df.groupby('UserId'):
            if len(user_reviews) >= 3:
                features = self.extract_user_features(user_reviews)
                user_features_list.append(features)
                user_ids.append(user_id)
        user_features_df = pd.DataFrame(user_features_list, index=user_ids)
        trust_labels = self.create_trust_labels(user_features_df)
        X = user_features_df.fillna(user_features_df.mean())
        y = trust_labels
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        self.model.fit(X_train_scaled, y_train)
        self.feature_names = list(X.columns)
        y_pred = self.model.predict(X_test_scaled)
        accuracy = accuracy_score(y_test, y_pred)
        print(f"Trust prediction accuracy: {accuracy:.3f}")
        print(f"Trained on {len(user_features_list)} users")
        return accuracy

    def predict_user_trust(self, user_reviews_df):
        features = self.extract_user_features(user_reviews_df)
        feature_df = pd.DataFrame([features])
        feature_df = feature_df.reindex(columns=self.feature_names, fill_value=0)
        features_scaled = self.scaler.transform(feature_df)
        trust_prob = self.model.predict_proba(features_scaled)[0][1]
        return trust_prob


In [ ]:
trust_predictor = UserTrustScorePredictor()
if len(df) > 0:
    trust_accuracy = trust_predictor.train_trust_model(df.head(5000))
else:
    print("No data found to train trust model.")


In [ ]:
class TrustAwareSentimentAnalysis:
    def __init__(self, sentiment_model, sentiment_tokenizer, trust_predictor):
        self.sentiment_model = sentiment_model
        self.sentiment_tokenizer = sentiment_tokenizer
        self.trust_predictor = trust_predictor
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.sentiment_model.to(self.device)

    def predict_sentiment(self, text):
        inputs = self.sentiment_tokenizer(text, return_tensors='pt', truncation=True, padding='max_length', max_length=128)
        inputs = {name: tensor.to(self.device) for name, tensor in inputs.items()}
        with torch.no_grad():
            outputs = self.sentiment_model(**inputs)
            probabilities = torch.softmax(outputs.logits, dim=1)
            positive_prob = probabilities[0][1].item()
        return positive_prob

    def predict_with_trust(self, review_text, user_reviews_df, trust_weight=0.3):
        base_sentiment = self.predict_sentiment(review_text)
        user_trust = self.trust_predictor.predict_user_trust(user_reviews_df)
        trust_weighted_sentiment = base_sentiment * (trust_weight * user_trust + (1 - trust_weight))
        return {
            'base_sentiment': base_sentiment,
            'user_trust_score': user_trust,
            'trust_weighted_sentiment': trust_weighted_sentiment,
            'trust_level': 'High' if user_trust > 0.7 else 'Medium' if user_trust > 0.4 else 'Low'
        }


In [ ]:
# Initialize analyzer (requires sentiment_model and sentiment_tok to be defined from training)
# If sentiment_model is not trained in this session, instruct user to load pre-trained folder.
if 'sentiment_model' in globals() and 'sentiment_tok' in globals():
    trust_aware_analyzer = TrustAwareSentimentAnalysis(sentiment_model, sentiment_tok, trust_predictor)
    sample_users = df.groupby('UserId').filter(lambda x: len(x) >= 5).groupby('UserId').head(1)
    for idx, (_, sample_review) in enumerate(sample_users.head(3).iterrows()):
        user_id = sample_review['UserId']
        user_reviews = df[df['UserId'] == user_id]
        result = trust_aware_analyzer.predict_with_trust(sample_review['Text'], user_reviews)
        print(f"User {idx+1} (ID: {user_id}): Base sentiment {result['base_sentiment']:.3f}, trust {result['user_trust_score']:.3f}, level {result['trust_level']}")
else:
    print("Sentiment model not available in this session. Load or train sentiment_model first.")


In [ ]:
# Save trust predictor locally (pickle)
with open('trust_predictor.pkl', 'wb') as f:
    pickle.dump(trust_predictor, f)
print("Saved trust_predictor.pkl")

# Save sentiment model and tokenizer folder if available
if 'sentiment_model' in globals() and 'sentiment_tok' in globals():
    sentiment_model.save_pretrained('./sentiment_model')
    sentiment_tok.save_pretrained('./sentiment_model')
    print("Saved sentiment_model/ folder")
else:
    print("Sentiment model not saved: train or load it first.")


In [ ]:
print("Training notebook done. Large files (sentiment_model/, .pkl) should be hosted externally (Google Drive, Hugging Face, etc.) and not committed to GitHub.")
